In [1]:
# Problem Statement
# Identify charges for insurance based on the metrics provided.
# After looking into the dataset its numerical data. 
# Both input and outputs are available.
# Output label is numerical.
###################################################
# Domain - MACHINE LEARNING
# Supervised Learning
# Regression
###################################################

In [2]:
import pandas as pd

In [3]:
dataset = pd.read_csv("insurance_pre.csv")
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
# Dataset has 1338 Rows X 6 Columns
# We have to convert the following nominal fields to numerical using one-hot encoding.
# The nominal fields are smoker, sex.

In [5]:
# Preprocessing
dataset = pd.get_dummies(dataset, dtype="int64", drop_first=True)
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [6]:
#  Display Columns
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [7]:
# Identify independent variables
independent = dataset[["age", "bmi", "children", "sex_male", "smoker_yes"]]
independent

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [8]:
# Identify dependent variable
dependent = dataset[["charges"]]
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [9]:
# Import GridSearchCV to automatically test different hyperparameter combinations
# using cross-validation and identify the best combination
from sklearn.model_selection import GridSearchCV

# Import DecisionTreeRegressor for performing Decision Tree Regression
from sklearn.tree import DecisionTreeRegressor

# Define the hyperparameters and the values that GridSearchCV should test
param_grid = {
    # Define different criteria used to measure the quality of a split
    # 'mse' and 'mae' are outdated names
    # mse => squared_error
    # mae => absolute_error
    'criterion': ['squared_error', 'absolute_error', 'friedman_mse'],

    # Define how many features should be considered when looking for the best split
    # None => consider all features
    # sqrt => consider sqrt(number of features)
    # log2 => consider log2(number of features)
    'max_features': [None, 'sqrt', 'log2'],

    # Define how the tree should choose the split
    # best => choose the best split
    # random => choose the best split among randomly selected splits
    'splitter': ['best', 'random']
}

# Create the GridSearchCV object
# DecisionTreeRegressor() is the model we want to tune
# param_grid contains all hyperparameter combinations to test
# refit=True means the best model will be trained again using the complete training data
# verbose=3 displays detailed progress information
# n_jobs=-1 uses all available CPU cores to speed up the search
grid = GridSearchCV(
    DecisionTreeRegressor(),
    param_grid,
    refit=True,
    verbose=3,
    n_jobs=-1
)

# Start the hyperparameter search using the training data
# GridSearchCV will train and evaluate the Decision Tree for every combination
grid.fit(independent, dependent)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,estimator,DecisionTreeRegressor()
,param_grid,"{'criterion': ['squared_error', 'absolute_error', ...], 'max_features': [None, 'sqrt', ...], 'splitter': ['best', 'random']}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'absolute_error'


In [10]:
# Get all results generated by GridSearchCV
# cv_results_ contains information about every hyperparameter combination tested,
# including parameters, training scores, validation scores, fit time, and ranking
re = grid.cv_results_

# Convert the GridSearchCV results dictionary into a pandas DataFrame
# This makes the results easier to read, filter, sort, and analyze
table = pd.DataFrame.from_dict(re)

# Display the complete results table
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.031239,0.016023,0.008204,0.005102,squared_error,None,best,"{'criterion': 'squared_error', 'max_features':...",0.718143,0.608060,0.725673,0.736886,0.675035,0.692759,0.047250,7
1,0.025309,0.009805,0.011099,0.005619,squared_error,None,random,"{'criterion': 'squared_error', 'max_features':...",0.694198,0.533531,0.750714,0.675398,0.697611,0.670290,0.072822,10
2,0.047331,0.019232,0.016374,0.007604,squared_error,sqrt,best,"{'criterion': 'squared_error', 'max_features':...",0.667711,0.621835,0.731490,0.661064,0.655924,0.667605,0.035652,11
3,0.029898,0.004063,0.014416,0.010313,squared_error,sqrt,random,"{'criterion': 'squared_error', 'max_features':...",0.664820,0.482201,0.723355,0.622742,0.694064,0.637436,0.084442,14
4,0.033305,0.009584,0.026004,0.005372,squared_error,log2,best,"{'criterion': 'squared_error', 'max_features':...",0.747454,0.568510,0.667049,0.500609,0.698152,0.636355,0.089592,16
5,0.035755,0.002716,0.016195,0.008551,squared_error,log2,random,"{'criterion': 'squared_error', 'max_features':...",0.684701,0.544405,0.491804,0.658491,0.523993,0.580679,0.076555,18
6,0.201354,0.020845,0.012966,0.008066,absolute_error,None,best,"{'criterion': 'absolute_error', 'max_features'...",0.717214,0.586440,0.692626,0.716510,0.744826,0.691523,0.055080,8
7,0.111908,0.033966,0.012512,0.007821,absolute_error,None,random,"{'criterion': 'absolute_error', 'max_features'...",0.662766,0.677856,0.688756,0.719560,0.755894,0.700966,0.033182,5
8,0.108354,0.032869,0.010663,0.006851,absolute_error,sqrt,best,"{'criterion': 'absolute_error', 'max_features'...",0.689111,0.648348,0.713598,0.604901,0.674528,0.666097,0.037194,12
9,0.101099,0.022616,0.015668,0.009221,absolute_error,sqrt,random,"{'criterion': 'absolute_error', 'max_features'...",0.703230,0.660435,0.809605,0.754434,0.675183,0.720578,0.054865,1


In [13]:
# Import r2_score to evaluate how well the model predicts the target values
from sklearn.metrics import r2_score

# Use the best model found by GridSearchCV to make predictions on the test dataset
# grid.predict() automatically uses grid.best_estimator_
grid_predictions = grid.predict(independent)

# Calculate the R2 score by comparing the actual test values (y_test)
# with the predictions made by the best model
r_score = r2_score(dependent, grid_predictions)

# Print the best hyperparameter combination found by GridSearchCV
# along with its R2 score on the unseen test dataset
print("The R2 value for best params {}: ".format(grid.best_params_), r_score)

The R2 value for best params {'criterion': 'absolute_error', 'max_features': 'sqrt', 'splitter': 'random'}:  0.8570180570322737


In [14]:
# Get all the results generated by GridSearchCV
# cv_results_ contains the parameters tested, CV scores, rankings,
# fitting time, and other information for every combination
re = grid.cv_results_

# Use the best model found by GridSearchCV to predict the test data
# grid.predict() automatically uses the best estimator after GridSearchCV
grid_predictions = grid.predict(independent)

# Display the predictions made for each row in the test dataset
grid_predictions

array([16884.924 ,  1725.5523,  5253.524 , ...,  1629.8335,  2007.945 ,
       29141.3603], shape=(1338,))

In [15]:
age = float(input("Age: "))
bmi = float(input("BMI: "))
children = float(input("Children: "))
sex = float(input("Sex Male 0 or 1: "))
smoker = float(input("Smoker Yes 0 or 1: "))
future_prediction = grid.predict(
    [[age, bmi, children, sex, smoker]]
)
future_prediction

Age:  32
BMI:  7.5
Children:  1
Sex Male 0 or 1:  0
Smoker Yes 0 or 1:  0


C:\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(


array([4766.022])